# Experiment Tracking on Snowflake

**Session 2 -- Deep Dive: Feature Engineering, Feature Stores & Experiment Tracking**  
**Notebook 3 of 4** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **ExperimentTracking API** | Initialize experiments and manage runs |
| **XGBoost with Autologging** | Use SnowflakeXgboostCallback for automatic metric/param/model logging |
| **Manual Runs** | Vary hyperparameters and log each run with artifacts |
| **Artifact Logging** | Save confusion matrices, feature importance plots, and classification reports |
| **Model Logging** | Log models directly to the Model Registry from experiment runs |
| **Run Comparison** | Query and compare metrics across all runs |

---

## Snowflake ExperimentTracking API

| API Method | Purpose |
|-----------|---------|
| `exp.set_experiment()` | Create or connect to a named experiment |
| `exp.start_run()` | Start a new run (context manager supported) |
| `exp.log_param()` | Log a single parameter (or `log_params` for batch) |
| `exp.log_metric()` | Log a metric (supports `step` for epoch tracking) |
| `exp.log_model()` | Log model to the Model Registry |
| `exp.log_artifact()` | Log files or directories (visible in Artifacts tab) |

### Key Advantage

There is **no separate tracking server** to deploy, secure, or scale.
Experiments are stored as Snowflake metadata objects within your database/schema,
governed by the same RBAC as everything else. You can view all experiments in
**Snowsight** under **AI & ML --> Experiments**.

### Prerequisites

Run notebooks 01 and 02 first. We will load the training dataset created in notebook 02.

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
%autoreload
import os
import tempfile

import pandas as pd
from xgboost import XGBClassifier

from session2_features_and_experiments.experiment_utils import (
    evaluate,
    get_model_params,
    log_run_artifacts,
    prepare_data,
    run_experiment,
)
from utils import get_config
from utils import get_feature_config, get_session

config = get_config("config.yaml")

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse

session = get_session(
    connection_name=config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)

print(f"Connected as {session.get_current_user()} | Role: {session.get_current_role()}")
print(
    f"Context: {session.get_current_database()}.{session.get_current_schema()} | Warehouse: {session.get_current_warehouse()}"
)

## 2 | Load Training Data from Feature Store Dataset

We load the immutable dataset created in notebook 02 and the test table.
Because the dataset is versioned and immutable, every run in this experiment
uses the **exact same data** -- making comparisons fair and reproducible.

In [ ]:
from snowflake.ml.dataset import load_dataset

# == Get the latest of the dataset ==
DATASET_NAME = "PATIENT_RISK_TRAINING_DS"
versions = session.sql(f"SHOW VERSIONS IN DATASET {DB}.{SCHEMA}.{DATASET_NAME}").collect()
ds_version = versions[-1]["version"]

# == Load the dataset version ==
dataset = load_dataset(session, f"{DB}.{SCHEMA}.{DATASET_NAME}", version=ds_version)
train_pdf = dataset.read.to_pandas()

### 2.1 Preprocess Data

In [ ]:
feature_config = get_feature_config(config)

# == Run preprocessing function ==
data = prepare_data(
    session=session,
    feature_config=feature_config,
    db=DB,
    schema=SCHEMA,
    train_pdf=train_pdf,
)

X_train_processed = data.X_train
X_test_processed = data.X_test
y_train_enc = data.y_train_enc
y_test_enc = data.y_test_enc
le = data.label_encoder

## 3 | Initialize Experiment Tracking

We create (or connect to) a named experiment. All runs will be grouped under this
experiment for easy comparison.

The experiment is stored as a Snowflake metadata object in your database/schema --
no external tracking server needed.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

# from snowflake.ml.model.model_signature import infer_signature

exp = ExperimentTracking(
    session=session,
    database_name=DB,
    schema_name=SCHEMA,
)

EXPERIMENT_NAME = "PATIENT_RISK_EXPERIMENT"
MODEL_NAME = "PATIENT_RISK_MODEL_DEMO"

# == NOTE: The only reason we delete the experiment here ==
# == is to have a clean demonstration in this notebook. ==
try:
    exp.delete_experiment(EXPERIMENT_NAME)
except:
    print("No experiment found. Continuing...")

exp.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Location: {DB}.{SCHEMA}")

## 4 | Baseline Run: XGBoost with Default Parameters + Autologging

The baseline establishes a reference point. We use the **SnowflakeXgboostCallback**
to automatically log parameters, per-iteration metrics, and the final model.

After training, we manually log artifacts:
- Confusion matrix plot
- Feature importance plot
- Classification report (text)

These will appear in the **Artifacts** tab of the run in Snowsight.

In [ ]:
artifact_dir = tempfile.mkdtemp(prefix="exp_artifacts_")

baseline_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.3,
    objective="multi:softmax",
    num_class=len(le.classes_),
    random_state=42,
    eval_metric="mlogloss",
)

RUN_NAME = "baseline_xgb_100"
with exp.start_run(RUN_NAME):
    # == Train the model ==
    baseline_model.fit(
        X_train_processed,
        y_train_enc,
        eval_set=[(X_test_processed, y_test_enc)],
        verbose=False,
    )

    # == Log Params ==
    exp.log_params(get_model_params(baseline_model))

    # == Calculate & Log Metrics ==
    baseline_metrics = evaluate(baseline_model, X_test_processed, y_test_enc, le)
    exp.log_metrics(baseline_metrics)

    # == Log step loss ==
    results = baseline_model.evals_result()
    for epoch, loss in enumerate(results["validation_0"]["mlogloss"]):
        exp.log_metric(key="validation_0:mlogloss", value=loss, step=epoch)

    # == Log any artifact files ==
    log_run_artifacts(exp, baseline_model, X_test_processed, y_test_enc, le, artifact_dir, RUN_NAME)

    print("Baseline metrics:")
    for k, v in baseline_metrics.items():
        print(f"  {k}: {v:.4f}")

## 6 | Experiment Run 2: More Trees + Lower Learning Rate

Hypothesis: increasing `n_estimators` to 200 and lowering `learning_rate` to 0.1
may improve generalization. We use autologging again plus manual artifact logging.

In [ ]:
run2_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(le.classes_),
    random_state=42,
    eval_metric="mlogloss",
)

RUN_NAME = "xgb_200_lr01"
with exp.start_run(RUN_NAME):
    run2_metrics = run_experiment(
        exp=exp,
        model=run2_model,
        experiment_name=RUN_NAME,
        train_features=X_train_processed,
        train_labels=y_train_enc,
        test_features=X_test_processed,
        test_labels=y_test_enc,
        le=le,
    )

## 7 | Experiment Run 3: Deeper Trees with Regularization

Hypothesis: `max_depth=10` with `reg_alpha=0.1` and `reg_lambda=1.0`
allows more complex splits while preventing overfitting via L1/L2 regularization.

In [ ]:
run3_model = XGBClassifier(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="multi:softmax",
    num_class=len(le.classes_),
    random_state=42,
    eval_metric="mlogloss",
)

RUN_NAME = "xgb_deep_regularized"
with exp.start_run(RUN_NAME):
    run3_metrics = run_experiment(
        exp=exp,
        model=run3_model,
        experiment_name=RUN_NAME,
        train_features=X_train_processed,
        train_labels=y_train_enc,
        test_features=X_test_processed,
        test_labels=y_test_enc,
        le=le,
    )

## 8 | Experiment Run 4: High-Performance Configuration

More trees (`n_estimators=300`), moderate depth, lower learning rate,
and subsampling (`colsample_bytree=0.8`, `subsample=0.8`) for a
robust gradient boosting configuration.

In [ ]:
run4_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    colsample_bytree=0.8,
    subsample=0.8,
    objective="multi:softmax",
    num_class=len(le.classes_),
    random_state=42,
    eval_metric="mlogloss",
)

RUN_NAME = "xgb_300_highperf"
with exp.start_run(RUN_NAME):
    run4_metrics = run_experiment(
        exp=exp,
        model=run4_model,
        experiment_name=RUN_NAME,
        train_features=X_train_processed,
        train_labels=y_train_enc,
        test_features=X_test_processed,
        test_labels=y_test_enc,
        le=le,
    )

## 9 | Compare All Runs

We can query the experiment runs using SQL (`SHOW RUNS IN EXPERIMENT`) and display
a comparison table.

You can also view this in **Snowsight** under **AI & ML --> Experiments**, where
you will now see the **Artifacts** tab populated with plots and reports for each run.

In [ ]:
runs_df = session.sql(f"SHOW RUNS IN EXPERIMENT {DB}.{SCHEMA}.{EXPERIMENT_NAME}").to_pandas()

print(f"Total runs in experiment '{EXPERIMENT_NAME}': {len(runs_df)}")
print()
display(runs_df)

In [ ]:
all_results = {
    "baseline_xgb_100": baseline_metrics,
    "xgb_200_lr01": run2_metrics,
    "xgb_deep_regularized": run3_metrics,
    "xgb_300_highperf": run4_metrics,
}

comparison = pd.DataFrame(all_results).T
comparison.index.name = "Run"
comparison = comparison.round(4)

print("Performance Comparison:")
print("=" * 80)
display(comparison)
print()

best_run = comparison["f1_weighted"].idxmax()
best_f1 = comparison.loc[best_run, "f1_weighted"]
print(f"Best run by f1_weighted: {best_run} ({best_f1:.4f})")

## 10 | Identify the Best Model

We select the run with the highest `f1_weighted` score. This model will be
registered in the Model Registry in notebook 04.

Note: Each run has already logged its model to the registry via the autolog callback.
You can also view artifacts (confusion matrices, feature importance plots, classification
reports) in the Snowsight Experiments UI under the **Artifacts** tab for each run.

In [ ]:
models = {
    "baseline_xgb_100": baseline_model,
    "xgb_200_lr01": run2_model,
    "xgb_deep_regularized": run3_model,
    "xgb_300_highperf": run4_model,
}

best_model = models[best_run]
best_metrics = all_results[best_run]

print(f"Best model: {best_run}")
print(f"Type: {type(best_model).__name__}")
print(f"Metrics:")
for k, v in best_metrics.items():
    print(f"  {k}: {v:.4f}")
print()
print("This model will be registered in the Model Registry in notebook 04.")

## 11 | Retrieve Artifacts Programmatically

You can also retrieve artifacts from a run using the Python API or SQL.
This is useful for downloading plots, reports, or model files outside of the UI.

In [ ]:
artifacts = exp.list_artifacts(run_name="BASELINE_XGB_100")
print("Artifacts in baseline run:")
for a in artifacts:
    print(f"  {a}")

print()
print("SQL equivalent:")
print(f"  LIST snow://experiment/{EXPERIMENT_NAME}/versions/BASELINE_XGB_100/;")
print(
    f"  GET snow://experiment/{EXPERIMENT_NAME}/versions/BASELINE_XGB_100/plots/confusion_matrix_baseline.png file:///tmp;"
)

## 12 | Summary

In this notebook we:

1. **Loaded** the immutable training dataset from the Feature Store
2. **Initialized** Snowflake's ExperimentTracking API
3. **Trained 4 XGBoost variations** with different hyperparameters
4. **Used autologging** (`SnowflakeXgboostCallback`) to automatically log params, per-iteration metrics, and models
5. **Logged artifacts** -- confusion matrix plots, feature importance charts, and classification reports
6. **Logged models** to the Model Registry via the autolog callback
7. **Compared** all runs side-by-side and identified the best performer

### Key Takeaways

| Aspect | How Snowflake Handles It |
|--------|--------------------------|
| Infrastructure | No server -- metadata lives in Snowflake |
| Artifact storage | `exp.log_artifact()` -- visible in Snowsight Artifacts tab |
| Autologging | `SnowflakeXgboostCallback` -- automatic params, metrics per iteration, model |
| Model logging | `exp.log_model()` / autolog callback wraps the Model Registry |
| Authentication | Same Snowflake credentials as your data |
| Governance | Snowflake RBAC (same as tables and views) |
| UI | Snowsight AI & ML tab |

### Logs Tab

The **Logs** tab in Snowsight captures stdout/stderr output but only works from
**Snowflake Notebooks** (Snowsight) or SPCS workloads. To enable it:

```python
exp.set_live_logging_status(True)
```

When running locally, use `exp.log_artifact()` to log text files with any
output you want to capture.

### Variables Passed to Notebook 4

- `best_model` -- the trained XGBClassifier
- `best_run` -- the run name
- `best_metrics` -- the evaluation metrics dict
- `X_train_processed`, `X_test_processed`, `y_test_enc` -- preprocessed train/test data
- `le` -- LabelEncoder for decoding predictions
- `preprocessor` -- fitted ColumnTransformer
- `EXPERIMENT_NAME` -- for querying experiment later

---

**Next -->** [04_model_registry_integration.ipynb](04_model_registry_integration.ipynb) -- Register the best model, version it, trace lineage, and run a promotion workflow